# 25 — Paper 3: Sensitivity analyses (NB tambon fixed effects, ปีงบประมาณ) → Table S2 และ Sect. 3.7

โน้ตบุ๊กนี้สร้างผล **sensitivity analyses (i)–(vi), (viii) และ (ix)** ของ Methods 2.11 ใหม่ทั้งหมด ด้วยข้อมูลและกติกาเดียวกับ **Notebook 23**
(ช่วง ม.ค. 2565 – พ.ย. 2567 ตัด ธ.ค., PM2.5 ดาวเทียมถ่วงประชากร, ERA5, ปีงบประมาณ ต.ค.–ก.ย.)

| # | Sensitivity (Methods 2.11) | ในโน้ตบุ๊กนี้ |
|---|---|---|
| — | แบบจำลองหลักแบบ FE (เทียบกับ NB-BYM2 ของ Notebook 23) | `Main (FE)` lag 0, lag 1, lag 0–1 |
| (i) | ตัด chronic rhinitis (J31.0) ออกจากกลุ่มทางเดินหายใจ | `(i) Excl. chronic rhinitis` |
| (ii) | PM2.5 ถ่วงพื้นที่แทนถ่วงประชากร | `(ii) Area-weighted PM2.5` |
| (iii) | ตัด 12 ตำบลที่มีโรงพยาบาลรัฐ | `(iii) Excl. hospital tambons` + ผลต่อ IRR ระหว่างตำบล |
| (iv) | รวมเป็นระดับอำเภอ | `(iv) District level` (district FE) |
| (v) | ตัดปีงบ 2565 (FY2022: ม.ค.–ก.ย. 2565 ในช่วงหลัก) | `(v) Excl. FY2022` |
| (vi) | buffer ริมถนน 150 m และ 300 m | effect modification แบบ frequentist (downwind + roadside groups) ที่ 150/300/500 m |
| (viii) | ทางเดินหายใจเฉพาะ COPD exacerbation (J44.0–J44.1) + asthma + pneumonia และแยกรายโรค | `(viii) …` (เพิ่ม 13 พ.ย. 2569) |
| (ix) | ischaemic heart disease (I20–I24) และ stroke (I60–I64) แยกกัน | `(ix) …` (เพิ่ม 13 พ.ย. 2569) |
| — | calendar-month FE แทน month-of-year + fiscal year | `Calendar-month FE` |
| (vii) | ขยายถึง ก.ย. 2569 + รวมแก่งคอย/ตาลเดี่ยว | **อยู่ใน Notebook 24** (ไม่ทำซ้ำที่นี่) |

**แบบจำลอง (Sect. 2.11):** negative binomial (NB2) พร้อม tambon fixed effects ซึ่งประมาณค่าสัมประสิทธิ์ภายในตำบลตัวเดียวกับ $\beta_W$ ใน NB-BYM2

$$\ln\mu_{it}=\ln E_{it}+\beta_W\,x^W_{it}+\mathbf z_t^\top\boldsymbol\gamma+\delta_{m(t)}+\tau_{fy(t)}+\alpha_i$$

- IRR ระหว่างตำบล (สำหรับ (iii)) มาจากแบบจำลอง **pooled** ที่ไม่มีผลรายตำบล: $\beta_W x^W_{it}+\beta_B x^B_i$ (Begg & Parides 2003)
- SE แบบ model-based (เหมือน Notebook 20) — ตั้ง `COV_TYPE = "cluster"` ถ้าต้องการ CI แบบ cluster-robust ตามตำบล
- (viii) และ (ix): ตำบลที่ไม่มี visit ของโรคนั้นเลยตลอดช่วงไม่มีข้อมูลต่อ FE model จึงตัดออกก่อน fit (ค่า IRR ไม่เปลี่ยน แต่ `n_units` บอกจำนวนตำบลที่ใช้จริง)
- รันทั้งโน้ตบุ๊ก ≈ 2–5 นาที (ไม่มี MCMC)

**ไฟล์ผลลัพธ์ใน `OUTDIR`:** `TableS2_sensitivity_long.csv`, `TableS2_sensitivity.csv` (wide), `TableS2_buffers_EM.csv`,
`Hospital_exclusion_between_within.csv`, `FigS_sensitivity.png`, `sensitivity_numbers.txt` (ข้อความตัวเลขสำหรับย่อหน้าท้าย Sect. 3.7)

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available, and outputs are
# written to results/nb25/ of the clone. Set USE_REPO = False to use the original Google Drive paths in CONFIG.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
if USE_REPO and REPO_DIR and os.path.isdir(os.path.join(REPO_DIR, "data", "processed")):
    os.environ["PAPER3_DATA_DIR"] = os.path.join(REPO_DIR, "data", "processed")
    os.environ["PAPER3_OUTDIR"] = os.path.join(REPO_DIR, "results", "nb25")
    print("Repository data:", os.environ["PAPER3_DATA_DIR"], "| outputs:", os.environ["PAPER3_OUTDIR"])
else:
    print("Repository not found - using the Google Drive paths in CONFIG")

## 0 · CONFIG

In [ ]:
# ---- ไฟล์ข้อมูล (เหมือน Notebook 23; ไม่พบ → ค้นชื่อไฟล์อัตโนมัติใน DATA_DIR และ MyDrive) ----
DATA_DIR        = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
PM_PANEL_CSV    = "pm25_tambon_monthly_fy2565_2569.csv"      # Notebook 06: ตำบล × เดือน
BOUNDARY_GPKG   = "sat_pm25_tambon_summary.gpkg"             # Notebook 06: ขอบเขต 111 ตำบล + ประชากร WorldPop 2020
HDC_MONTHLY_CSV = "000_research/11HealthImpactAssessment_SW/0data/hdc_saraburi_pm25_disease_tambon_monthly_2565_2569.csv"
DIW_CSV         = "3-ColabProjects/Paper2/02_Data/processed/diw_facilities_geocoded.csv"
ROADSIDE_CSV    = "roadside_tambon.csv"                       # Notebook 08
ERA5_CACHE_DIR  = "ERA5_cache"
ERA5_POINT      = (14.50, 101.00)
NB23_TABLE5_LONG = "../../results/nb23/Table5_IRR_long.csv"   # repo: results/nb23/ · Drive: found by file name  # เทียบ FE กับ BYM2

# ---- ผลลัพธ์ ----
OUTDIR = "/content/drive/MyDrive/3-ColabProjects/Paper3/04_Reproduced_tables/25_Sensitivity_FE_FiscalYear"

# ---- ช่วงวิเคราะห์และตัวแปร (ต้องตรงกับ Notebook 23) ----
START, END      = "2022-01", "2024-11"
DROP_DECEMBER   = True
PM_COL          = "pm25_sat_pop"       # หลัก: ถ่วงประชากร
PM_AREA_COL     = None                 # (ii) ถ่วงพื้นที่: None = หาอัตโนมัติ (คอลัมน์ที่มีคำว่า area)
PM_UNIT         = 10
GROUPS = {
    "Respiratory":    ["COPD", "Asthma", "Pneumonia", "Influenza", "Acute pharyngitis", "Chronic rhinitis", "Bronchitis"],
    "Cardiovascular": ["Ischemic heart diseases", "Cerebrovascular diseases (Stroke)"],
    "Eye":            ["กลุ่มโรคตาอักเสบ"],
    "Skin":           ["กลุ่มโรคผิวหนังอักเสบ"],
}
RHINITIS        = "Chronic rhinitis"
# (viii) และ (ix) — คอลัมน์โรคจาก HDC (เพิ่ม 13 พ.ย. 2569)
SUBSET_OUTCOMES = {
    "Resp acute (COPD+asthma+pneumonia)": ["COPD", "Asthma", "Pneumonia"],
    "COPD only": ["COPD"], "Asthma only": ["Asthma"], "Pneumonia only": ["Pneumonia"],
    "Stroke only": ["Cerebrovascular diseases (Stroke)"], "IHD only": ["Ischemic heart diseases"],
}
LAGS = {"Lag 0": [0], "Lag 1": [1], "Lag 0–1": [0, 1]}
DOWNWIND_SECTOR, DOWNWIND_KM = (247.5, 337.5), 5.0   # CPF 67.5–157.5° กลับทิศ — ต้องตรงกับ Notebook 23 (แก้ 10 พ.ย. 2569, เดิม 270–315°)
ROADSIDE_COL    = "share_500m_MAIN"
BUFFERS_M       = [150, 300, 500]      # (vi) — คอลัมน์ share_<m>m_MAIN ใน ROADSIDE_CSV
EXCLUDE_FY      = 2565                 # (v) ปีงบ พ.ศ. 2565 = FY2022 (ต.ค. 2564 – ก.ย. 2565)

# ---- (iii) โรงพยาบาลรัฐ 12 แห่ง (รหัสหน่วยบริการ 5 หลัก) ----
HOSPITAL_HOSPCODES = {
    "10661": "รพ.สระบุรี",   "10695": "รพ.พระพุทธบาท", "10807": "รพ.แก่งคอย",  "10808": "รพ.หนองแค",
    "10809": "รพ.วิหารแดง",  "10810": "รพ.หนองแซง",    "10811": "รพ.บ้านหมอ",  "10812": "รพ.ดอนพุด",
    "10813": "รพ.หนองโดน",   "10814": "รพ.เสาไห้ เฉลิมพระเกียรติ ๘๐ พรรษา", "10815": "รพ.มวกเหล็ก", "10816": "รพ.วังม่วง",
}
# ตำบลที่ HDC ลงรหัสพื้นที่ให้แต่ละโรงพยาบาล (ช่วง 2565–2567) — หาอัตโนมัติจาก MOPH Open Data (s_pm25_month) ในหัวข้อ 2
# ถ้า Colab เชื่อมต่อ Open Data ไม่ได้ ให้ใส่เองเป็น {hospcode: "รหัสตำบล 6 หลัก"} (ใช้ areacode 8 หลักตัด 6 หลักแรก)
# ดึงจาก MOPH Open Data s_pm25_month (hospcode × areacode) ปี พ.ศ. 2565 และ 2567 เมื่อ 10 พ.ย. 2569 → hospcode_areacode_saraburi.csv
# รพ.แก่งคอย 10807 = 19020108 (แก่งคอย) ถึงปี 2567, เปลี่ยนเป็น 19020308 (ตาลเดี่ยว) ปี 2568 · รพ.เสาไห้ 10814 = 19100107 (2565) / 19100102 (2567) — ตำบลเดียวกัน
HOSPITAL_TAMBON = {
    "10661": "190101", "10695": "190903", "10807": "190201", "10808": "190301", "10809": "190402", "10810": "190506",
    "10811": "190601", "10812": "190701", "10813": "190801", "10814": "191001", "10815": "191102", "10816": "191203",
}
HDC_LOOKUP_YEAR = 2567                 # ปีปฏิทิน พ.ศ. ที่ใช้ดึงคู่ hospcode → areacode
HOSPITAL_CODES  = []                   # (ทางเลือกสุดท้าย) รหัสตำบล 6 หลักโดยตรง — ข้ามการหาจาก hospcode
N_HOSPITAL      = 12

COV_TYPE        = "nonrobust"          # "nonrobust" (เหมือน Notebook 20) หรือ "cluster" (cluster-robust ตามตำบล)

In [ ]:
import importlib.util, subprocess, sys
need = {"statsmodels": "statsmodels", "pyproj": "pyproj", "openpyxl": "openpyxl"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, re, glob, time, warnings
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, requests
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats, optimize
from pyproj import Geod
warnings.filterwarnings("ignore"); pd.set_option("display.width", 180)
UTM, GEOD = "EPSG:32647", Geod(ellps="WGS84")

IN_COLAB = os.path.isdir("/content")
DATA_DIR = os.environ.get("PAPER3_DATA_DIR", DATA_DIR)
if IN_COLAB and DATA_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
OUTDIR = os.environ.get("PAPER3_OUTDIR", OUTDIR)
MYDRIVE = "/content/drive/MyDrive" if IN_COLAB else DATA_DIR
os.makedirs(OUTDIR, exist_ok=True)

def find(p, must=True):
    """path เต็ม → ใช้เลย · path สัมพัทธ์ → DATA_DIR, MyDrive · ไม่พบ → ค้นชื่อไฟล์ใน DATA_DIR และ MyDrive (ลึก 4 ชั้น)"""
    if os.path.isabs(p) and os.path.exists(p): return p
    for b in (DATA_DIR, MYDRIVE, "."):
        if os.path.exists(os.path.join(b, p)): return os.path.join(b, p)
    name = os.path.basename(p)
    for base in dict.fromkeys((DATA_DIR, MYDRIVE)):
        for root, dirs, files in os.walk(base):
            if root[len(base):].count(os.sep) >= 4: dirs[:] = []
            if name in files: return os.path.join(root, name)
    if must: raise FileNotFoundError(f"ไม่พบ {p} — แก้ path ใน CONFIG")
    return None
print("statsmodels", sm.__version__, "| OUTDIR =", OUTDIR)

## 1 · ข้อมูล (โค้ดเดียวกับ Notebook 23 หัวข้อ 1–4)
นอกจากนี้อ่านรหัสหน่วยบริการจากไฟล์ HDC (ถ้ามี) เพื่อระบุตำบลที่มีโรงพยาบาล

In [ ]:
code6 = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str.zfill(6).str[:6]

BND = gpd.read_file(find(BOUNDARY_GPKG)); BND["a_code"] = code6(BND["a_code"])
_dc = next((c for c in BND.columns if c.lower() in
       ("district", "district_en", "district_name", "amphoe", "amphoe_en", "amp_name", "amp_name_e", "adm2_en", "name_2")), None)
BND["district_name"] = BND[_dc].astype(str) if _dc else BND.a_code.str[:4]
BND = BND[["a_code", "name", "district_name", "pop", "geometry"]].drop_duplicates("a_code").reset_index(drop=True)

PM = pd.read_csv(find(PM_PANEL_CSV), encoding="utf-8-sig", dtype={"a_code": str}); PM["a_code"] = code6(PM.a_code)
PM["ym"] = PM.ym.astype(str).str[:7]
if PM_COL not in PM: raise KeyError(f"ไม่มีคอลัมน์ {PM_COL} ใน {PM_PANEL_CSV}: {list(PM.columns)}")
if PM_AREA_COL is None:
    cand = [c for c in PM.columns if "area" in c.lower() and "pm" in c.lower() and pd.api.types.is_numeric_dtype(PM[c])]
    PM_AREA_COL = cand[0] if cand else None
print("คอลัมน์ PM2.5:", [c for c in PM.columns if "pm" in c.lower()], "| (ii) ใช้:", PM_AREA_COL or "ไม่พบ → ข้าม (ii)")

ALL_DIS = [d for v in GROUPS.values() for d in v]
H0 = pd.read_csv(find(HDC_MONTHLY_CSV), encoding="utf-8-sig", dtype=str)
H0.columns = [c.strip() for c in H0.columns]
ALIAS = {"Ischemic heart disease": "Ischemic heart diseases", "Stroke": "Cerebrovascular diseases (Stroke)",
         "Eye inflammation": "กลุ่มโรคตาอักเสบ", "Dermatitis": "กลุ่มโรคผิวหนังอักเสบ",
         "r1_1": "COPD", "r1_2": "Asthma", "r1_3": "Pneumonia", "r1_4": "Influenza", "r1_5": "Acute pharyngitis",
         "r1_6": "Chronic rhinitis", "r1_7": "Bronchitis", "r2_1": "Ischemic heart diseases",
         "r2_2": "Cerebrovascular diseases (Stroke)", "r3": "กลุ่มโรคตาอักเสบ", "r4": "กลุ่มโรคผิวหนังอักเสบ"}
H0 = H0.rename(columns={k: v for k, v in ALIAS.items() if k in H0.columns and v not in H0.columns})
miss = [d for d in ALL_DIS if d not in H0]
if miss: raise KeyError(f"ไม่พบคอลัมน์โรค {miss} ใน HDC — คอลัมน์ที่มี: {list(H0.columns)}")
H0["a_code"] = code6(H0.a_code)
if "ym" not in H0:
    y = pd.to_numeric(H0.year_be); y = np.where(y > 2400, y - 543, y)
    H0["ym"] = [f"{int(a):04d}-{int(float(b)):02d}" for a, b in zip(y, H0.month)]
H0["ym"] = H0.ym.astype(str).str[:7]
for d in ALL_DIS: H0[d] = pd.to_numeric(H0[d], errors="coerce").fillna(0)
t_ = pd.to_datetime(H0.ym)
H0["fy"] = t_.dt.year + 543 + (t_.dt.month >= 10).astype(int)
H0 = H0[H0.a_code.str[:2] == "19"]
FAC_COL = next((c for c in H0.columns if c.lower() in ("hospcode", "hoscode", "hcode", "hosp_code", "off_id", "facility_code", "hospital_code")), None)
H = H0.groupby(["a_code", "ym", "fy"], as_index=False)[ALL_DIS].sum()
TAM_CODES = sorted(set(BND.a_code) & set(H.a_code))
print(f"ตำบลที่ใช้ {len(TAM_CODES)} | คอลัมน์รหัสหน่วยบริการใน HDC: {FAC_COL or 'ไม่มี'}")

In [ ]:
# ERA5 รายเดือน (เหมือน Notebook 23 หัวข้อ 2)
def era5_hourly(lat, lon, year):
    cdir = find(ERA5_CACHE_DIR, must=False) or os.path.join(DATA_DIR, ERA5_CACHE_DIR)
    os.makedirs(cdir, exist_ok=True)
    hits = sorted(glob.glob(os.path.join(cdir, f"era5_{lat:.2f}_{lon:.2f}_{year}_*.csv.gz")), key=os.path.getmtime)
    if hits: return pd.read_csv(hits[-1], parse_dates=["time"])
    end = min(pd.Timestamp(f"{year}-12-31"), pd.Timestamp.today().normalize() - pd.Timedelta(days=6))
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=180, params=dict(
        latitude=lat, longitude=lon, start_date=f"{year}-01-01", end_date=f"{end:%Y-%m-%d}", models="era5",
        hourly="temperature_2m,relative_humidity_2m,precipitation", timezone="Asia/Bangkok"))
    r.raise_for_status(); df = pd.DataFrame(r.json()["hourly"]); df["time"] = pd.to_datetime(df.time)
    df.to_csv(os.path.join(cdir, f"era5_{lat:.2f}_{lon:.2f}_{year}_openmeteo.csv.gz"), index=False)
    return df

ERA = pd.concat([era5_hourly(*ERA5_POINT, y) for y in range(int(START[:4]), int(END[:4]) + 1)], ignore_index=True)
ERA["ym"] = ERA.time.dt.strftime("%Y-%m")
MET = ERA.groupby("ym").agg(temp=("temperature_2m", "mean"), rh=("relative_humidity_2m", "mean"),
                            precip=("precipitation", "sum")).reset_index()
MET = MET[(MET.ym >= START) & (MET.ym <= END)]
print(f"ERA5: {len(MET)} เดือน")

In [ ]:
# ตัวแปรระดับตำบล: downwind และกลุ่มริมถนน (ทุก buffer) — เหมือน Notebook 23 หัวข้อ 3
TAM = BND[BND.a_code.isin(TAM_CODES)].sort_values("a_code").reset_index(drop=True); N = len(TAM)
pt_ = TAM.to_crs(UTM).representative_point().to_crs(4326); TLAT, TLON = pt_.y.values, pt_.x.values
DIW = pd.read_csv(find(DIW_CSV)).dropna(subset=["lat", "lon"])
lo_, hi_ = DOWNWIND_SECTOR; dw = np.zeros(N, bool)
for la, lo in zip(DIW.lat, DIW.lon):
    az_, _, d = GEOD.inv(np.full(N, lo), np.full(N, la), TLON, TLAT); az_ = az_ % 360
    dw |= (d / 1000 <= DOWNWIND_KM) & (az_ >= lo_) & (az_ <= hi_)
TAM["downwind"] = dw.astype(int)

RS = pd.read_csv(find(ROADSIDE_CSV), encoding="utf-8-sig", dtype={"a_code": str}); RS["a_code"] = code6(RS.a_code)
BUF_COLS = {}
for b in BUFFERS_M:
    col = ROADSIDE_COL.replace("500", str(b))
    if col not in RS: print(f"⚠ ไม่มีคอลัมน์ {col} ใน {ROADSIDE_CSV} → ข้าม buffer {b} m"); continue
    BUF_COLS[b] = col
    s = TAM[["a_code"]].merge(RS[["a_code", col]], on="a_code", how="left")[col].fillna(0).values
    med = np.median(s[s > 0]) if (s > 0).any() else 0
    TAM[f"road_{b}"] = np.where(s == 0, "none", np.where(s <= med, "low", "high"))
    print(f"buffer {b} m: {pd.Series(TAM[f'road_{b}']).value_counts().to_dict()}")
TAM["road_grp"] = TAM["road_500"] if "road_500" in TAM else "none"
print(f"downwind {TAM.downwind.sum()} จาก {N} ตำบล")

In [ ]:
# ตาราง ตำบล × เดือน (เหมือน Notebook 23 หัวข้อ 4) + PM2.5 ถ่วงพื้นที่
def lagged(col):
    P = PM[PM.a_code.isin(TAM_CODES)][["a_code", "ym", col]].rename(columns={col: "pm"})
    P["per"] = pd.PeriodIndex(P.ym, freq="M"); out = None
    for name_, ks in LAGS.items():
        m_ = None
        for k in ks:
            s = P.assign(per=P.per + k); s["ym"] = s.per.astype(str); s = s[["a_code", "ym", "pm"]].rename(columns={"pm": f"_l{k}"})
            m_ = s if m_ is None else m_.merge(s, on=["a_code", "ym"])
        m_[name_] = m_[[f"_l{k}" for k in ks]].mean(axis=1, skipna=False); m_ = m_[["a_code", "ym", name_]]
        out = m_ if out is None else out.merge(m_, on=["a_code", "ym"], how="outer")
    return out

PMX = lagged(PM_COL)
if PM_AREA_COL: PMX = PMX.merge(lagged(PM_AREA_COL)[["a_code", "ym", "Lag 0"]].rename(columns={"Lag 0": "Lag 0 area"}),
                                on=["a_code", "ym"], how="left")
road_cols = [c for c in TAM.columns if c.startswith("road_")]
D = (H[H.a_code.isin(TAM_CODES)].merge(PMX, on=["a_code", "ym"], how="left")
     .merge(MET, on="ym", how="inner").merge(TAM[["a_code", "pop", "downwind"] + road_cols], on="a_code"))
D = D[(D.ym >= START) & (D.ym <= END)]
if DROP_DECEMBER: D = D[~D.ym.str.endswith("-12")]
D = D.dropna(subset=list(LAGS)).sort_values(["a_code", "ym"]).reset_index(drop=True)
for g, cols in GROUPS.items(): D[g] = D[cols].sum(axis=1).round().astype(int)
D["Respiratory excl. rhinitis"] = D[[c for c in GROUPS["Respiratory"] if c != RHINITIS]].sum(axis=1).round().astype(int)
# (viii) ชุดที่น่าจะเป็นอาการเฉียบพลันมากที่สุด และแยกรายโรค · (ix) IHD แยกจาก stroke
for name_, cols_ in SUBSET_OUTCOMES.items(): D[name_] = D[cols_].sum(axis=1).round().astype(int)
D["moy"] = pd.to_datetime(D.ym).dt.month; D["district"] = D.a_code.str[:4]
for v in ["temp", "rh", "precip"]: D["z_" + v] = (D[v] - D[v].mean()) / D[v].std()
print(f"{len(D):,} ตำบล-เดือน = {D.a_code.nunique()} ตำบล × {D.ym.nunique()} เดือน | ปีงบ {D.groupby('fy').ym.nunique().to_dict()}")
print("จำนวนครั้ง:", {g: int(D[g].sum()) for g in GROUPS})

## 2 · ตำบลที่มีโรงพยาบาลรัฐ (สำหรับ (iii))
ลำดับการหา: `HOSPITAL_CODES` → `HOSPITAL_TAMBON` → **ดึงคู่ hospcode → areacode ของ 12 โรงพยาบาลจาก MOPH Open Data** → รหัสหน่วยบริการในไฟล์ HDC → (สำรอง) 12 ตำบลที่มีจำนวนครั้งสูงสุด
ใช้ตำบลที่ **HDC ลงรหัสไว้** ไม่ใช่ที่อยู่จริง (เช่น รพ.แก่งคอยถูกลงรหัสตำบลแก่งคอยจนถึง ธ.ค. 2567 — Sect. 2.2)
**ตรวจ:** 12 ตำบล, สัดส่วน visits ≈ 71.8% / 78.8%, เป็นตำบลที่ตั้งอำเภอ (รหัสลงท้าย 01) 7 ตำบล (Sect. 2.2)

In [ ]:
def hdc_hosp_to_tambon(hospcodes, year_be):
    """ดึงแถวดิบ s_pm25_month (hospcode × areacode) ของสระบุรี แล้วคืน {hospcode: รหัสตำบล 6 หลัก}"""
    rows, off = [], 0
    while True:
        r = requests.post("https://opendata.moph.go.th/api/report_data", timeout=(15, 120),
                          headers={"User-Agent": "curl/8", "Content-Type": "application/json"},
                          json={"tableName": "s_pm25_month", "year": str(year_be), "province": "19", "type": "json",
                                "limit": 10000, "offset": off})
        r.raise_for_status(); js = r.json(); data = js.get("data", []) if isinstance(js, dict) else js
        rows += data
        if not data or len(rows) >= int(js.get("total", len(rows)) if isinstance(js, dict) else len(rows)): break
        off += 10000
    raw = pd.DataFrame(rows); hc = next(c for c in raw.columns if c.lower() in ("hospcode", "hoscode", "hcode"))
    raw["hc"] = raw[hc].astype(str).str.zfill(5); raw["a6"] = raw["areacode"].astype(str).str.zfill(8).str[:6]
    m = raw[raw.hc.isin(hospcodes)].groupby("hc").a6.agg(lambda s: s.value_counts().index[0])
    multi = raw[raw.hc.isin(hospcodes)].groupby("hc").a6.nunique(); multi = multi[multi > 1]
    if len(multi): print("⚠ hospcode ที่มีหลายรหัสพื้นที่ (ใช้ตัวที่พบบ่อยสุด):", multi.to_dict())
    return m.to_dict()

tot = D.groupby("a_code")[list(GROUPS)].sum().sum(axis=1).sort_values(ascending=False)
H2T = {}
if not HOSPITAL_CODES:
    H2T = dict(HOSPITAL_TAMBON)
    if len(H2T) < len(HOSPITAL_HOSPCODES):
        try:
            H2T = {**hdc_hosp_to_tambon(list(HOSPITAL_HOSPCODES), HDC_LOOKUP_YEAR), **H2T}
            print(f"✓ ดึงคู่ hospcode → ตำบล จาก MOPH Open Data (ปี {HDC_LOOKUP_YEAR})")
        except Exception as ex:
            print(f"⚠ ดึงจาก MOPH Open Data ไม่ได้ ({type(ex).__name__}: {str(ex)[:80]}) — Colab อาจถูกบล็อก IP ต่างประเทศ\n"
                  f"  → ใส่ HOSPITAL_TAMBON ใน CONFIG เอง หรือรันเซลล์นี้บนเครื่องในประเทศไทย")
    miss = [h for h in HOSPITAL_HOSPCODES if h not in H2T]
    if miss: print("⚠ ยังไม่ทราบตำบลของ:", {h: HOSPITAL_HOSPCODES[h] for h in miss})
    H2T = {h: str(t).zfill(6)[:6] for h, t in H2T.items()}
if HOSPITAL_CODES:
    HOSP = sorted(code6(pd.Series(HOSPITAL_CODES))); HOSP_SRC = "CONFIG (HOSPITAL_CODES)"
elif len(H2T) == len(HOSPITAL_HOSPCODES):
    HOSP = sorted(set(H2T.values()) & set(TAM_CODES)); HOSP_SRC = "hospcode ของโรงพยาบาลรัฐ 12 แห่ง"
    nm = TAM.set_index("a_code").name
    display(pd.DataFrame([(h, HOSPITAL_HOSPCODES[h], t, nm.get(t, "ไม่อยู่ในตำบลที่วิเคราะห์")) for h, t in H2T.items()],
                         columns=["hospcode", "โรงพยาบาล", "a_code", "ตำบล"]))
elif FAC_COL:
    f = H0[[FAC_COL, "a_code", "ym"]].dropna(); f[FAC_COL] = f[FAC_COL].str.replace(r"\D", "", regex=True)
    f = f[(f[FAC_COL].str.len() <= 5) & (f[FAC_COL].str.zfill(5).str[0] == "1")]
    f = f[f.ym.isin(D.ym.unique())]
    HOSP = sorted(set(f.a_code) & set(TAM_CODES)); HOSP_SRC = f"รหัสหน่วยบริการ ({FAC_COL})"
else:
    HOSP = sorted(tot.index[:N_HOSPITAL]); HOSP_SRC = f"⚠ {N_HOSPITAL} ตำบลที่มีจำนวนครั้งสูงสุด (ไม่มีรหัสหน่วยบริการ — ตรวจรายชื่อ)"
D["hospital"] = D.a_code.isin(HOSP).astype(int)
ht = TAM[TAM.a_code.isin(HOSP)][["a_code", "name", "district_name"]].assign(visits=lambda x: x.a_code.map(tot).astype(int))
print(f"ตำบลโรงพยาบาล {len(HOSP)} ตำบล — ที่มา: {HOSP_SRC}")
print(f"สัดส่วนจำนวนครั้งทั้งหมด {tot[HOSP].sum() / tot.sum():.1%} | ทางเดินหายใจ "
      f"{D[D.hospital == 1].Respiratory.sum() / D.Respiratory.sum():.1%}  (manuscript: 71.8% / 78.8%)")
display(ht.sort_values("visits", ascending=False))
print(f"เป็นตำบลที่ตั้งอำเภอ (รหัสลงท้าย 01): {sum(h.endswith('01') for h in HOSP)} จาก {len(HOSP)} (manuscript: 7 จาก 12)")
if len(HOSP) != N_HOSPITAL: print(f"⚠ ได้ {len(HOSP)} ตำบล ไม่ใช่ {N_HOSPITAL} — ตรวจ HOSPITAL_TAMBON / HOSPITAL_CODES")

## 3 · ฟังก์ชันแบบจำลอง
- `add_x()` คำนวณ $x^W$ และ $x^B$ ใหม่ในทุกชุดข้อมูลย่อย (ค่าเฉลี่ยตำบลจากแถวที่ใช้จริง)
- `fit_nb()` — NB2: ประมาณ α แบบ profile maximum likelihood แล้ว fit GLM-NB ที่ α นั้น (เหมือน `MASS::glm.nb`) — ใช้วิธีเดียวกันทุกแบบจำลอง

In [ ]:
Z = ["z_temp", "z_rh", "z_precip"]
def add_x(df, pm="Lag 0", unit="a_code"):
    df = df.copy(); m = df.groupby(unit)[pm].transform("mean")
    df["xw"] = (df[pm] - m) / PM_UNIT
    um = df.groupby(unit)[pm].mean(); df["xb"] = (m - um.mean()) / PM_UNIT
    return df

def fit_nb(df, outcome, terms, fe="a_code", time="moy+fy"):
    d = df.copy(); d["yy"] = d[outcome].astype(float)
    d["off"] = np.log(d["pop"] * d.yy.sum() / d["pop"].sum())
    tt = ["C(moy)", "C(fy)"] + Z if time == "moy+fy" else ["C(ym)"]
    f = "yy ~ " + " + ".join(terms + tt + ([f"C({fe})"] if fe else []))
    ckw = dict(cov_type="cluster", cov_kwds={"groups": pd.factorize(d[fe or "a_code"])[0]}) if COV_TYPE == "cluster" else {}
    # NB2: α ประมาณด้วย maximum likelihood แบบ profile (เหมือน MASS::glm.nb ใน R) แล้ว fit GLM-NB ที่ α นั้น
    # (รุ่นแรกใช้ α แบบ moment เมื่อ discrete NB ไม่ลู่เข้า → α ต่ำเกินจริง → CI แคบเกินไป; แก้ 10 พ.ย. 2569)
    model = lambda a: smf.glm(f, data=d, family=sm.families.NegativeBinomial(alpha=a), offset=d.off)
    cache = {}
    def nll(la):
        k = round(la, 6)
        if k not in cache: cache[k] = -model(np.exp(la)).fit(start_params=cache.get("p")).llf
        return cache[k]
    p0 = smf.glm(f, data=d, family=sm.families.Poisson(), offset=d.off).fit(); cache["p"] = p0.params.values
    opt = optimize.minimize_scalar(nll, bounds=(np.log(1e-5), np.log(50)), method="bounded", options={"xatol": 1e-3})
    alpha = float(np.exp(opt.x))
    r = model(alpha).fit(**ckw)
    if not np.all(np.isfinite(r.bse[terms])): raise RuntimeError("SE ไม่จำกัด")
    edge = opt.x <= np.log(1e-5) + 0.01 or opt.x >= np.log(50) - 0.01
    r.alpha_, r.how_, r.n_ = alpha, "NB2 profile-ML" + (" (α ชนขอบ)" if edge else ""), (len(d), d[fe or "a_code"].nunique(), d.ym.nunique())
    return r

def eff(r, term="xw", combo=None):
    """IRR ต่อ 10 µg/m³ ของ term หรือผลรวมเชิงเส้น combo={term: weight}"""
    combo = combo or {term: 1.0}; names = list(combo); w = np.array([combo[k] for k in names])
    b = float(w @ r.params[names].values); V = r.cov_params().loc[names, names].values; se = float(np.sqrt(w @ V @ w))
    return dict(IRR=np.exp(b), lo=np.exp(b - 1.96 * se), hi=np.exp(b + 1.96 * se), p=2 * stats.norm.sf(abs(b / se)))
fmt = lambda r: f"{r['IRR']:.2f} ({r['lo']:.2f}–{r['hi']:.2f})"

## 4 · Sensitivity (i)–(v), (viii), (ix) และ calendar-month FE → Table S2
ทุกแถวคือ IRR **ภายในตำบล** ต่อ PM2.5 10 µg/m³ ที่ lag 0 (ยกเว้นแถว lag ของแบบจำลองหลัก) จาก NB tambon FE

In [ ]:
SCEN = [("Main (FE), lag 0",                dict()),
        ("Main (FE), lag 1",                dict(pm="Lag 1")),
        ("Main (FE), lag 0–1",              dict(pm="Lag 0–1")),
        ("(i) Excl. chronic rhinitis",      dict(outcomes={"Respiratory": "Respiratory excl. rhinitis"})),
        ("(ii) Area-weighted PM2.5",        dict(pm="Lag 0 area")),
        ("(iii) Excl. hospital tambons",    dict(rows=lambda d: d.hospital == 0)),
        ("(iv) District level",             dict(level="district")),
        (f"(v) Excl. FY2022 (FY{EXCLUDE_FY} BE)", dict(rows=lambda d: d.fy != EXCLUDE_FY)),
        ("(viii) COPD + asthma + pneumonia", dict(outcomes={"Respiratory": "Resp acute (COPD+asthma+pneumonia)"}, drop_zero=True)),
        ("(viii) COPD exacerbation only",    dict(outcomes={"Respiratory": "COPD only"}, drop_zero=True)),
        ("(viii) Asthma only",               dict(outcomes={"Respiratory": "Asthma only"}, drop_zero=True)),
        ("(viii) Pneumonia only",            dict(outcomes={"Respiratory": "Pneumonia only"}, drop_zero=True)),
        ("(ix) Stroke only",                 dict(outcomes={"Cardiovascular": "Stroke only"}, drop_zero=True)),
        ("(ix) Ischaemic heart disease only", dict(outcomes={"Cardiovascular": "IHD only"}, drop_zero=True)),
        ("Calendar-month FE",               dict(time="ym"))]

def to_district(df, outcome, pm):
    g = df.assign(_w=df[pm] * df["pop"]).groupby(["district", "ym", "fy", "moy"] + Z, as_index=False).agg(
        y=(outcome, "sum"), pop=("pop", "sum"), _w=("_w", "sum"))
    g[pm] = g._w / g["pop"]; g[outcome] = g.y
    return g

rows = []
for label, kw in SCEN:
    pmv = kw.get("pm", "Lag 0")
    if pmv not in D or D[pmv].isna().all(): print(f"– ข้าม {label}: ไม่มีคอลัมน์ {pmv}"); continue
    base = D[kw["rows"](D)] if "rows" in kw else D
    base = base.dropna(subset=[pmv])
    for g in GROUPS:
        oc = kw.get("outcomes", {}).get(g, g)
        if "outcomes" in kw and g not in kw["outcomes"]: continue           # (i), (viii), (ix) ใช้เฉพาะกลุ่มที่ระบุ
        t0 = time.time()
        bg = base[base.groupby("a_code")[oc].transform("sum") > 0] if kw.get("drop_zero") else base   # (viii), (ix)
        try:
            if kw.get("level") == "district":
                dd = add_x(to_district(bg, oc, pmv), pmv, unit="district"); r = fit_nb(dd, oc, ["xw"], fe="district")
            else:
                r = fit_nb(add_x(bg, pmv), oc, ["xw"], time=kw.get("time", "moy+fy"))
            e = eff(r); rows.append(dict(scenario=label, outcome=g, **e, alpha=r.alpha_, fit=r.how_,
                                         n_rows=r.n_[0], n_units=r.n_[1], n_months=r.n_[2]))
        except Exception as ex:
            print(f"✗ {label} / {g}: {type(ex).__name__}: {str(ex)[:120]}"); continue
    print(f"✓ {label}: " + " | ".join(f"{r_['outcome'][:4]} {fmt(r_)}" for r_ in rows if r_["scenario"] == label), flush=True)

S2 = pd.DataFrame(rows)
S2.to_csv(f"{OUTDIR}/TableS2_sensitivity_long.csv", index=False, encoding="utf-8-sig")
S2W = S2.assign(v=S2.apply(fmt, axis=1)).pivot(index="scenario", columns="outcome", values="v")
S2W = S2W.reindex(index=[s for s, _ in SCEN if s in S2W.index], columns=[g for g in GROUPS if g in S2W.columns]).fillna("–")
S2W.to_csv(f"{OUTDIR}/TableS2_sensitivity.csv", encoding="utf-8-sig")
display(S2W)

### 4b · เทียบ FE กับ NB-BYM2 (Notebook 23)
Methods 2.11 เขียนว่า "the main analysis gave nearly identical estimates under both approaches" — ตารางนี้ใช้ตรวจข้อความนั้น

In [ ]:
p23 = find(NB23_TABLE5_LONG, must=False)
if p23:
    B = pd.read_csv(p23, encoding="utf-8-sig"); B = B[B.term.str.startswith("Lag")]
    F = S2[S2.scenario.str.startswith("Main (FE)")].assign(term=lambda x: x.scenario.str.replace("Main (FE), l", "L", regex=False))
    CMP = B.merge(F, on=["outcome", "term"], suffixes=("_BYM2", "_FE"))
    CMP["BYM2"] = CMP.apply(lambda r: f"{r.IRR_BYM2:.2f} ({r.lo_BYM2:.2f}–{r.hi_BYM2:.2f})", axis=1)
    CMP["FE"] = CMP.apply(lambda r: f"{r.IRR_FE:.2f} ({r.lo_FE:.2f}–{r.hi_FE:.2f})", axis=1)
    CMP["diff_%"] = (CMP.IRR_FE / CMP.IRR_BYM2 - 1) * 100
    CMP[["outcome", "term", "BYM2", "FE", "diff_%"]].to_csv(f"{OUTDIR}/FE_vs_BYM2.csv", index=False, encoding="utf-8-sig")
    display(CMP[["outcome", "term", "BYM2", "FE", "diff_%"]].round(1))
else:
    CMP = None; print("ไม่พบ Table5_IRR_long.csv ของ Notebook 23 — ข้ามการเทียบ")

## 5 · (iii) ตัดตำบลโรงพยาบาล: IRR ระหว่างตำบล vs ภายในตำบล
แบบจำลอง pooled (ไม่มีผลรายตำบล) ให้ทั้ง $\beta_B$ และ $\beta_W$ · แบบ FE ให้ $\beta_W$ — ใช้กับประโยค "Excluding the hospital tambons changed the pooled between-tambon IRR by …"

In [ ]:
hrows = []
for g in GROUPS:
    for lab, sub in [("All tambons", D), ("Excl. hospital tambons", D[D.hospital == 0])]:
        dd = add_x(sub)
        rp = fit_nb(dd, g, ["xw", "xb"], fe=None); rf = fit_nb(dd, g, ["xw"])
        hrows.append(dict(outcome=g, sample=lab, n_tambon=dd.a_code.nunique(),
                          **{f"between_pooled_{k}": v for k, v in eff(rp, "xb").items()},
                          **{f"within_pooled_{k}": v for k, v in eff(rp, "xw").items()},
                          **{f"within_FE_{k}": v for k, v in eff(rf, "xw").items()}))
HX = pd.DataFrame(hrows); HX.to_csv(f"{OUTDIR}/Hospital_exclusion_between_within.csv", index=False, encoding="utf-8-sig")
chg = {}
for g in GROUPS:
    a, b = HX[(HX.outcome == g) & (HX["sample"] == "All tambons")].iloc[0], HX[(HX.outcome == g) & (HX["sample"] != "All tambons")].iloc[0]
    chg[g] = {k: (b[f"{k}_IRR"] / a[f"{k}_IRR"] - 1) * 100 for k in ["between_pooled", "within_pooled", "within_FE"]}
    chg[g].update({f"{k}_all": a[f"{k}_IRR"] for k in ["between_pooled", "within_FE"]})
    chg[g].update({f"{k}_excl": b[f"{k}_IRR"] for k in ["between_pooled", "within_FE"]})
display(pd.DataFrame(chg).T.round(3))

## 6 · (vi) Effect modification ที่ buffer 150 / 300 / 500 m (frequentist, NB tambon FE)
ความชันภายในตำบล = $\beta_W + \beta_D\,\text{downwind} + \beta_L\,\text{low} + \beta_H\,\text{high}$ (เหมือน Notebook 23 หัวข้อ 9 แต่เป็น FE)

In [ ]:
erows = []
for b in BUF_COLS:
    dd = add_x(D); grp = dd[f"road_{b}"]
    dd["xw_dw"] = dd.xw * dd.downwind; dd["xw_lo"] = dd.xw * (grp == "low"); dd["xw_hi"] = dd.xw * (grp == "high")
    for g in GROUPS:
        r = fit_nb(dd, g, ["xw", "xw_dw", "xw_lo", "xw_hi"])
        combos = {"Not downwind, roadside none": {"xw": 1}, "Downwind (roadside none)": {"xw": 1, "xw_dw": 1},
                  "Roadside low": {"xw": 1, "xw_lo": 1}, "Roadside high": {"xw": 1, "xw_hi": 1}}
        for k, c in combos.items():
            erows.append(dict(buffer_m=b, outcome=g, group=k, **eff(r, combo=c),
                              n_tambon=int(dd.loc[(grp == k.split()[-1].lower()) if k.startswith("Roadside") else
                                                  (dd.downwind == 1) if k.startswith("Downwind") else dd.downwind == 0, "a_code"].nunique())))
        for t in ["xw_dw", "xw_lo", "xw_hi"]:
            erows.append(dict(buffer_m=b, outcome=g, group=f"interaction {t}", **eff(r, t)))
EMB = pd.DataFrame(erows); EMB.to_csv(f"{OUTDIR}/TableS2_buffers_EM.csv", index=False, encoding="utf-8-sig")
show = EMB[EMB.outcome == "Respiratory"].assign(v=lambda x: x.apply(fmt, axis=1) + x.p.map(lambda p: f" p={p:.2f}"))
display(show.pivot(index="group", columns="buffer_m", values="v"))

## 7 · Forest plot และตัวเลขสำหรับ Sect. 3.7

In [ ]:
R = S2[S2.outcome == "Respiratory"].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7, 0.45 * len(R) + 1)); yy = np.arange(len(R))[::-1]
ax.errorbar(R.IRR, yy, xerr=[R.IRR - R.lo, R.hi - R.IRR], fmt="o", capsize=3, color="#2c7bb6")
ax.set_yticks(yy, R.scenario); ax.axvline(1, ls="--", c="k")
ax.set_xlabel("Within-tambon respiratory IRR per 10 µg/m³ PM$_{2.5}$ (95% CI)")
plt.tight_layout(); plt.savefig(f"{OUTDIR}/FigS_sensitivity.png", dpi=300); plt.show()

L = []
lag0 = R[~R.scenario.str.startswith("Main") & ~R.scenario.str.startswith("(viii)")]
L.append(f"Respiratory, FE main lag 0: {fmt(R[R.scenario == 'Main (FE), lag 0'].iloc[0])}.")
L.append(f"Lag-0 sensitivity analyses (i)–(v) and calendar-month FE gave respiratory IRRs between "
         f"{lag0.IRR.min():.2f} and {lag0.IRR.max():.2f}.")
ex = lag0[(lag0.lo > 1) | (lag0.hi < 1)]
L.append("Intervals excluding 1: " + ("; ".join(f"{r.scenario} {fmt(r._asdict())}" for r in ex.itertuples()) if len(ex) else "none") + ".")
for r in lag0.itertuples(): L.append(f"  {r.scenario}: {fmt(r._asdict())}")
SUB = S2[S2.scenario.str.startswith(("(viii)", "(ix)"))]
L.append("Outcome subsets (viii)–(ix), tambons with any visits only:")
for r in SUB.itertuples(): L.append(f"  {r.scenario} [{r.outcome}]: {fmt(r._asdict())}; {r.n_units} tambons; alpha {r.alpha:.3g} ({r.fit})")
c = chg["Respiratory"]
L.append(f"Excluding the {len(HOSP)} hospital tambons ({HOSP_SRC}) changed the pooled between-tambon IRR by "
         f"{c['between_pooled']:+.0f}% (from {c['between_pooled_all']:.3g} to {c['between_pooled_excl']:.3g}) "
         f"but the within-tambon (FE) IRR by {c['within_FE']:+.1f}% (from {c['within_FE_all']:.3f} to {c['within_FE_excl']:.3f}).")
if CMP is not None:
    L.append(f"FE vs NB-BYM2 within-tambon IRR: max absolute difference {CMP['diff_%'].abs().max():.1f}% across groups and lags.")
for b in BUF_COLS:
    e = EMB[(EMB.outcome == "Respiratory") & (EMB.buffer_m == b)].set_index("group")
    L.append(f"Buffer {b} m (respiratory, FE): downwind {fmt(e.loc['Downwind (roadside none)'])}, not downwind/none {fmt(e.loc['Not downwind, roadside none'])}, "
             f"roadside low {fmt(e.loc['Roadside low'])}, high {fmt(e.loc['Roadside high'])}; "
             f"interaction p: downwind {e.loc['interaction xw_dw','p']:.2f}, low {e.loc['interaction xw_lo','p']:.2f}, high {e.loc['interaction xw_hi','p']:.2f}.")
L.append(f"Model: NB2 with tambon fixed effects, SE {COV_TYPE}; months {D.ym.min()}–{D.ym.max()} (December excluded), "
         f"{D.a_code.nunique()} tambons; PM2.5 area-weighted column: {PM_AREA_COL}.")
open(f"{OUTDIR}/sensitivity_numbers.txt", "w", encoding="utf-8").write("\n".join(L))
print("\n".join(L))

## 8 · ข้อความใน manuscript ที่ต้องตรวจกับผลนี้
- **Sect. 3.7 ย่อหน้าท้าย:** ช่วง IRR ของ sensitivity, analysis ที่ CI ไม่คร่อม 1, ผลการตัดตำบลโรงพยาบาล (between −x% vs within +y%)
- **Sect. 4 Discussion ย่อหน้า 2:** ประโยค "Excluding the hospital tambons changed the pooled between-tambon IRR by … but the within-tambon IRR by …"
- **Sect. 3.6:** "Frequentist interaction models gave the same pattern with 150 m and 300 m buffers" → ตรวจกับ `TableS2_buffers_EM.csv`
- **Methods 2.11:** "the main analysis gave nearly identical estimates under both approaches" → ตรวจกับ `FE_vs_BYM2.csv`
- **Methods 2.11 / Sect. 3.7 / Discussion ย่อหน้า 2:** ผล (viii) และ (ix) — ตรวจกับบรรทัด `Outcome subsets` ใน `sensitivity_numbers.txt` (13 พ.ย. 2569: 0.65, 0.97, 1.10, 0.58; stroke 0.92, IHD 1.00)
- **Online Resource 1, Table S2:** `TableS2_sensitivity.csv` + `TableS2_buffers_EM.csv` (+ ผล (vii) จาก Notebook 24)